In [2]:
import os
import re
import time
import requests
import datetime
from selenium import webdriver
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from bs4 import BeautifulSoup
from dotenv import load_dotenv
import pandas as pd
import numpy as np

In [3]:
# How to read_csv from google sheet https://medium.com/@Bwhiz/step-by-step-guide-importing-google-sheets-data-into-pandas-ae2df899257f
sheet_name = 'Agent' # replace with your own sheet name
sheet_id = '1QXAKFfXR8oBMXL_mVJx3SkqEpN0qIRG5davdcwCq1-0' # replace with your sheet's ID
url = f"https://docs.google.com/spreadsheets/d/{sheet_id}/gviz/tq?tqx=out:csv&sheet={sheet_name}"

agent_list = pd.read_csv(url)
agent_list.head()

,prefix_name,agent name,Unnamed: 2
0,property,Nut Ananganjanagit,NaN
1,asset,eakapong_r,NaN
2,condo,Pluto_Landmarks,NaN
3,estate,MaxTierRealty,NaN
4,prop,suwit,NaN


In [ ]:
def is_agent(fullname: str, prefixes: list) -> bool:
    normalized_fullname = fullname.replace(" ", "").lower()
    return any(prefix.replace(" ", "").lower() in normalized_fullname for prefix in prefixes)

In [347]:
def extract_numbers(text: str) -> str:
    """Extracts numbers from text while keeping hyphens and dot."""
    return "".join(re.findall(r"[-\d.]+", text))

In [348]:
agent_name_list = agent_list['agent name'].dropna().str.lower().tolist()
prefix_name = agent_list['prefix_name'].dropna().str.lower().tolist()

In [349]:
room_dict = {
    "date": [],
    "condo_name":[],
    "building": [],
    "floor": [],
    "area": [],
    "bedroom": [],
    "bathroom": [],
    "price": [],
    "phone": [],
    "owner": [],
    "room_number": [],
    "line_id": [],
    "url": []
}

In [350]:
page_urls = ["https://www.livinginsider.com/searchword/Condo/Rent/1/%E0%B8%A3%E0%B8%B1%E0%B8%87%E0%B8%AA%E0%B8%B4%E0%B8%95.html",
            ]

In [352]:
def search_room(url):    
    i = 0
    while True:

        page_content = requests.get(url)
        soup = BeautifulSoup(page_content.content, 'html.parser')
        posts = soup.find_all('div', class_="col-md-3 col-sm-4 col-xs-6 padding_topic margin_top_topic")
        # print(len(posts))
        for post in posts:

            link = post.find('a', class_="image-ratio-4-3").get('href')
            in_post_content = requests.get(link)
            in_post_soup = BeautifulSoup(in_post_content.content, 'html.parser')
            detail_content = in_post_soup.find('div', class_="detail-content col-sm-12 main-detail-content")
            owner = detail_content.find('label').text
            if owner.lower() in agent_name_list or is_agent(owner, prefix_name):
                print(f"Owner : {owner} continue")
                continue
            condo_name = detail_content.find('span', class_="text_project_detail_green").text
            price = detail_content.find('div', class_="t-24 price-detail mt-0 price_topic").text.replace('\n', '')
            
            detail_property = detail_content.find('div', class_="row detail-row")
            room_details = detail_property.find_all('span', class_="detail-property-list-text")
            properties = []
            for detail in room_details:
                room_detail = detail.text
                room_detail = re.sub(r'[\s\n]', '', room_detail)
                properties.append(room_detail)

            room_dict['date'].append(datetime.datetime.now().strftime("%d/%m/%Y"))
            room_dict['condo_name'].append(condo_name)
            room_dict['building'].append(None)
            room_dict['floor'].append(extract_numbers(properties[1]))
            room_dict['area'].append(extract_numbers(properties[0]))
            room_dict['bedroom'].append(extract_numbers(properties[2]))
            room_dict['bathroom'].append(extract_numbers(properties[3]))
            room_dict['price'].append(extract_numbers(price))
            room_dict['phone'].append(None)
            room_dict['owner'].append(owner)
            room_dict['room_number'].append(None)
            room_dict['line_id'].append(None)
            room_dict['url'].append(link)

        pagination = soup.find('ul', class_="pagination")
        pages = pagination.find_all('a', class_="loading_page")
        next_page = pages[-1]
        print(next_page.text)
        if next_page.text != " Next ›" or i == 50:
            print("Break!!!")
            break
        else:
            url = next_page.get('href')
            i += 1
            print(f"Finish page :{i}")

In [353]:
for page_url in page_urls:
    search_room(page_url)

Owner : Easyhomeland continue
Owner : NextHome continue
Owner : NextHome continue
Owner : เพื่อนอสังหาฯ... continue
Owner : PaphatAsset continue
Owner : bkkagencyproperty continue
Owner : Fast forrent condo continue
Owner : Unicondo continue
Owner : Differestate continue
Owner : Fast forrent condo continue
Owner : thammarat_pens continue
Owner : Fast forrent condo continue
Owner : rassamee_mnliving continue
Owner : Gowealthasset continue
Owner : matchaprop continue
Owner : บริการโพสต์_อสังหา... continue
Owner : serve1 continue
Owner : serve1 continue
 Next ›
Finish page :1
Owner : serve1 continue
Owner : serve1 continue
Owner : serve1 continue
Owner : serve1 continue
Owner : serve1 continue
Owner : serve1 continue
Owner : serve1 continue
Owner : serve1 continue
Owner : serve1 continue
Owner : serve1 continue
Owner : serve1 continue
Owner : qhome2188 continue
Owner : BoatProperty continue
Owner : katnuttakun continue
Owner : nuch.chaba64@gmail.c... continue
Owner : Wealthiness_Estate co

In [ ]:
OWNER_PATH = './csv_files/owner_room.csv'
EXPECTED_AGENT_PATH = './csv_files/expected_agent.csv'
EXPECTED_AGENT_LIST_PATH = './csv_files/expected_agent_list.csv'

all_room_df = pd.DataFrame(room_dict)

# filler owner
filltered_df = all_room_df[all_room_df.groupby('owner')['owner'].transform("count") < 4]

# fillter agent
expected_agent_df = all_room_df[all_room_df.groupby('owner')['owner'].transform("count") >= 4]
expected_agent_list = pd.Series(expected_agent_df['owner'].value_counts())

# export to csv
filltered_df.to_csv(OWNER_PATH)
expected_agent_df.to_csv(EXPECTED_AGENT_PATH)
expected_agent_list.to_csv(EXPECTED_AGENT_LIST_PATH)

In [355]:
filltered_df.groupby("owner").size()

owner
0834292959              1
ARMTRAIPOP              1
Airutdar Pholjun        1
Alisa_Uasaman           2
Apiwit                  3
                       ..
รัตนชาติ...             1
สุดชา ผาผง...           1
เกรดสุรางค์...          1
โอปอชา_ลาลาลันล่า...    3
ให้ทุกวันเป็นกำไร...    1
Length: 144, dtype: int64

In [356]:
expected_agent_df.groupby("owner").size()

owner
Charles-Heng              4
Chayapa                   6
Natchaaa                 15
Tanya_Noi                18
primporn                  5
ณัฏฐนันท์  (คุณสุ)...    12
dtype: int64

In [357]:
expected_agent_df

,date,condo_name,building,floor,area,bedroom,bathroom,price,phone,owner,room_number,line_id,url
1,10/03/2025,เคฟ คอนโด,None,24..,24..,4,1,12000.500...,None,Natchaaa,None,None,https://www.livinginsider.com/livingdetail/472...
11,10/03/2025,ดีคอนโด ไฮป์ รังสิต,None,26..,26..,5-10,1,12000.462...,None,Natchaaa,None,None,https://www.livinginsider.com/livingdetail/253...
16,10/03/2025,เทอร์ร่า เรสซิเดนซ์ เฟส 1 - 2,None,33..,33..,30,1,12000.364...,None,Natchaaa,None,None,https://www.livinginsider.com/livingdetail/239...
17,10/03/2025,เคฟ ทาวน์ ไอส์แลนด์,None,24.60..,24.60..,1,1,14000.569...,None,Natchaaa,None,None,https://www.livinginsider.com/livingdetail/238...
18,10/03/2025,เคฟ ทาวน์ ไอส์แลนด์,None,22..,22..,7,1,12000.545...,None,Natchaaa,None,None,https://www.livinginsider.com/livingdetail/238...
19,10/03/2025,นิว ครอส คูคต สเตชัน,None,30.55..,30.55..,5-10,1,11000.360...,None,primporn,None,None,https://www.livinginsider.com/livingdetail/223...
20,10/03/2025,นิว คอร์ คูคต สเตชัน,None,22.25..,22.25..,5-10,1,8500.382...,None,primporn,None,None,https://www.livinginsider.com/livingdetail/240...
21,10/03/2025,นิว คอร์ คูคต สเตชัน,None,46.50..,46.50..,1-4,2,20000.430...,None,primporn,None,None,https://www.livinginsider.com/livingdetail/245...
22,10/03/2025,นิว คอร์ คูคต สเตชัน,None,26.25..,26.25..,1-4,1,9500.362...,None,primporn,None,None,https://www.livinginsider.com/livingdetail/239...
23,10/03/2025,นิว ครอส คูคต สเตชัน,None,26.50..,26.50..,1-4,1,9000.340...,None,primporn,None,None,https://www.livinginsider.com/livingdetail/239...
